In [1]:
__author__ = "Nico Valencia"


# Purpose of this script is to create null models for Khat Use Frequency GWAS using POLMM
# https://github.com/WenjianBI/POLMM/blob/master/R/POLMM_Null_Model.R

In [1]:
# if you would like to view more command for POLMM (https://github.com/WenjianBI/POLMM), you
# can run this locally in R
# library(devtools)  # author version: 2.3.0
# install_github("WenjianBi/POLMM")
# library(POLMM)
# ?POLMM  # manual of POLMM() function with an example code, expected output and expected run time for demo

In [2]:
import hail as hl
import hailtop.batch as hb

# docker image
POLMM_IMAGE = "nsvalencia/polmm:v6"

# backend; billing
backend = hb.ServiceBackend(
    billing_project="neale-pumas-bge",
    remote_tmpdir="gs://neurogap-bge-imputed-regional/nico/tmp",
    regions=["us-central1"],
)

def get_file_size_gb(file):
    file_info = hl.utils.hadoop_stat(file)
    return file_info["size_bytes"] / (1024 * 1024 * 1024)

Loading BokehJS ...

/Users/nvalenci/.pyenv/versions/3.11.5/lib/python3.11/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:28: UserWarning:

You have specified the GCS requester pays configuration in both your spark-defaults.conf (/usr/local/Cellar/apache-spark/3.5.3/libexec/conf/spark-defaults.conf) and either an explicit argument or through `hailctl config`. For GCS requester pays configuration, Hail first checks explicit arguments, then `hailctl config`, then spark-defaults.conf.



In [2]:
# this section does a lot, but generally speaking it runs POLMM_Null_Model which
# looks like this equation:
# objNull <- POLMM_Null_Model(
#  formula = as.factor(assist_khat_amt) ~ age + sex +
#    PC1 + PC2 + PC3 + PC4 + PC5 +
#    PC6 + PC7 + PC8 + PC9 + PC10 +
#    study_site, --> indicates covariates
#  data = ph_m, --> the phenotype file
#  subjData = keep$IID, --> identifies the subject IDs (matched between .fam file and pheno file)
#  PlinkFile = plink_prefix, --> input plink files
#  control = list(LOCO = TRUE) --> indicate LOCO
#)
def fit_null_model_polmm_dense(
    b,
    site,
    bfile,
    phenotypes,
    phenotype_col,
    covariate_cols,
    storage,
    iid_col="IID",
):
# job name 
    j = b.new_job(name=f"polmm-step1-{site}-{phenotype_col}")

    j.image(POLMM_IMAGE)
    j.cpu(16)
    j.memory("16Gi")
    j.storage(f"{storage}Gi")

    j.declare_resource_group(null_model={"rds": "{root}.rds"})

    plink_prefix = f"{j.tmpdir}/plink/plink"
    r_script = f"{j.tmpdir}/run_polmm_null.R"

    j.command(f"""
set -euo pipefail

mkdir -p {j.tmpdir}/plink

cp {bfile['bed']} {plink_prefix}.bed
cp {bfile['bim']} {plink_prefix}.bim
cp {bfile['fam']} {plink_prefix}.fam

cat > {r_script} << 'RSCRIPT'
suppressPackageStartupMessages({{
  library(POLMM)
  library(data.table)
}})

args <- commandArgs(trailingOnly=TRUE)

plink_prefix <- args[1]
pheno_file <- args[2]
pheno_col <- args[3]
covars_csv <- args[4]
iid_col <- args[5]
out_rds <- args[6]

read_fam <- function(fam_file) {{
  fread(fam_file, col.names = c("FID","IID","PID","MID","SEX","PHENO"))
}}

clean_complete <- function(ph, iid_col, y_col, covars) {{
  needed <- c(iid_col, y_col, covars)
  needed <- intersect(needed, names(ph))
  keep_idx <- complete.cases(ph[, ..needed])
  message("Dropped ", sum(!keep_idx), " rows with missing phenotype/covariates")
  ph[keep_idx]
}}

fam <- read_fam(paste0(plink_prefix, ".fam"))
ph <- fread(pheno_file)

char_cols <- names(ph)[sapply(ph, is.character)]
for (cc in char_cols) {{
  ph[[cc]] <- trimws(ph[[cc]])
}}

covars <- trimws(strsplit(covars_csv, ",")[[1]])

ph[[pheno_col]] <- trimws(as.character(ph[[pheno_col]]))
ph[[pheno_col]][ph[[pheno_col]] %in% c("", "NA", ".", "-9")] <- NA
ph[[pheno_col]] <- as.integer(ph[[pheno_col]])

setkeyv(fam, "IID")
setkeyv(ph, iid_col)

keep <- fam[ph, nomatch = 0]
ph_m <- ph[match(keep$IID, ph[[iid_col]]), ]

stopifnot(all(keep$IID == ph_m[[iid_col]]))

ph_m <- clean_complete(ph_m, iid_col, pheno_col, covars)

keep <- keep[keep$IID %in% ph_m[[iid_col]]]
ph_m <- ph_m[match(keep$IID, ph_m[[iid_col]]), ]

stopifnot(all(keep$IID == ph_m[[iid_col]]))

message("Covariate classes:")
print(sapply(ph_m[, covars, with = FALSE], class))

rhs <- paste(covars, collapse = " + ")
fml <- as.formula(paste0("as.factor(", pheno_col, ") ~ ", rhs))

message("Formula: ", deparse(fml))
message("Subjects in phenotype after cleaning: ", nrow(ph_m))
message("Subjects in subjData: ", length(keep$IID))

objNull <- POLMM_Null_Model(
  formula = fml,
  data = ph_m,
  PlinkFile = plink_prefix,
  subjData = keep$IID,
  control = list(LOCO = TRUE)
)

saveRDS(objNull, out_rds)

message("Saved POLMM null model: ", out_rds)
RSCRIPT

Rscript {r_script} \\
  {plink_prefix} \\
  {phenotypes} \\
  {phenotype_col} \\
  "{covariate_cols}" \\
  {iid_col} \\
  {j.null_model['rds']}
""")

    return j.null_model

In [4]:
# Indicate sites, covariates, directory of plink files, phenotype files, and output
phenotype = "assist_khat_amt"

sites = [
    "KEMRI",
    "Moi",
    "AAU",
    "Uganda",
]

covariate_cols = (
    "age,"
    "sex,"
    "PC1,PC2,PC3,PC4,PC5,"
    "PC6,PC7,PC8,PC9,PC10,"
    "study_site"
)

base_path = "gs://neurogap-bge-imputed-regional/nico/khat_gwas"

plink_dir = (
    f"{base_path}/gp_missingness_dataset/gp08/gwaspy"
)

pheno_dir = (
    f"{base_path}/phenotype_files/"
    "special_char_change/no_hyphen/w_site/"
    "StudySite_w_MatchingIIDs"
)

output_path = (
    f"{base_path}/polmm_null/"
    "studySite_numeric_covars"
)

In [8]:
b = hb.Batch(
    backend=backend,
    name=f"POLMM dense null - {phenotype}",
)

jobs = {}

# plink files being used
for site in sites:
    plink_prefix = f"{plink_dir}/gp08_{site}_ldpruned"

    bfile = b.read_input_group(
        bed=f"{plink_prefix}.bed",
        bim=f"{plink_prefix}.bim",
        fam=f"{plink_prefix}.fam",
    )

    phenotypes = b.read_input(
        f"{pheno_dir}/no_char_{site}_khat_pheno_subset_lerato_with_studysite.tsv"
    )

    plink_size = round(get_file_size_gb(f"{plink_prefix}.bed"))
    storage = max(20, round(plink_size * 2) + 10)

    jobs[site] = fit_null_model_polmm_dense(
        b=b,
        site=site,
        bfile=bfile,
        phenotypes=phenotypes,
        phenotype_col=phenotype,
        covariate_cols=covariate_cols,
        storage=storage,
        iid_col="IID",
    )

# output written to: 
# gs://neurogap-bge-imputed-regional/nico/khat_gwas/polmm_null/studySite_numeric_covars
for site, null_model in jobs.items():
    b.write_output(
        null_model,
        f"{output_path}/gp08_{site}_{phenotype}_polmm_null",
    )

batch = b.run(wait=False)
batch

/var/folders/11/84m9v1z9659br_10hygjq6n80000gp/T/ipykernel_63902/611018845.py:13: DeprecationWarning:

Call to deprecated function (or staticmethod) hadoop_stat. (Prefer hailtop.fs.stat) -- Deprecated since version 0.2.137.



Output()

In [24]:
backend.close()

In [6]:
print(fit_null_model_polmm_dense)

<function fit_null_model_polmm_dense at 0x106cdb880>
